In [1]:
import pandas as pd

behaviors = pd.read_csv(
    "../data/raw/MINDsmall_train/behaviors.tsv",
    sep="\t",
    header=None,
    names=["impression_id", "user_id", "time", "history", "impressions"]
)

In [13]:
def parse_impressions(row):
    global malformed_count
    records = []
    if pd.isna(row["impressions"]):
        return records
    for token in row["impressions"].split(" "):
        try:
            news_id, click = token.split("-")
            click = int(click)
            if click not in (0, 1):
                raise ValueError
        except ValueError:
            malformed_count += 1
            continue
        records.append({...})
    return records

def parse_history(row):
    records = []
    if pd.isna(row["history"]):
        return records
    for news_id in row["history"].split(" "):
        records.append({
            "impression_id": row["impression_id"],
            "user_id": row["user_id"],
            "time": row["time"],
            "item_id": news_id,
            "click": 1,
            "source": "history",
        })
    return records

In [3]:
impression_rows = behaviors.apply(parse_impressions, axis=1).explode().dropna()
history_rows = behaviors.apply(parse_history, axis=1).explode().dropna()

In [5]:
interactions_impression = pd.DataFrame(list(impression_rows))
interactions_history = pd.DataFrame(list(history_rows))

display(interactions_impression.head())
display(interactions_history.head())

,impression_id,user_id,time,item_id,click,source
0,1,U13740,11/11/2019 9:05:58 AM,N55689,1,impression
1,1,U13740,11/11/2019 9:05:58 AM,N35729,0,impression
2,2,U91836,11/12/2019 6:11:30 PM,N20678,0,impression
3,2,U91836,11/12/2019 6:11:30 PM,N39317,0,impression
4,2,U91836,11/12/2019 6:11:30 PM,N58114,0,impression


,impression_id,user_id,time,item_id,click,source
0,1,U13740,11/11/2019 9:05:58 AM,N55189,1,history
1,1,U13740,11/11/2019 9:05:58 AM,N42782,1,history
2,1,U13740,11/11/2019 9:05:58 AM,N34694,1,history
3,1,U13740,11/11/2019 9:05:58 AM,N45794,1,history
4,1,U13740,11/11/2019 9:05:58 AM,N18445,1,history


In [6]:
interactions_history.to_parquet("../data/processed/interactions_history_train.parquet")
interactions_impression.to_parquet("../data/processed/interactions_impression_train.parquet")

In [7]:
behaviors_dev = pd.read_csv(
    "../data/raw/MINDsmall_dev/behaviors.tsv",
    sep="\t",
    header=None,
    names=["impression_id", "user_id", "time", "history", "impressions"]
)

impression_rows_dev = behaviors_dev.apply(parse_impressions, axis=1).explode().dropna()
history_rows_dev = behaviors_dev.apply(parse_history, axis=1).explode().dropna()

interactions_impression_dev = pd.DataFrame(list(impression_rows_dev))
interactions_history_dev = pd.DataFrame(list(history_rows_dev))

display(interactions_impression_dev.head())
display(interactions_history_dev.head())

,impression_id,user_id,time,item_id,click,source
0,1,U80234,11/15/2019 12:37:50 PM,N28682,0,impression
1,1,U80234,11/15/2019 12:37:50 PM,N48740,0,impression
2,1,U80234,11/15/2019 12:37:50 PM,N31958,1,impression
3,1,U80234,11/15/2019 12:37:50 PM,N34130,0,impression
4,1,U80234,11/15/2019 12:37:50 PM,N6916,0,impression


,impression_id,user_id,time,item_id,click,source
0,1,U80234,11/15/2019 12:37:50 PM,N55189,1,history
1,1,U80234,11/15/2019 12:37:50 PM,N46039,1,history
2,1,U80234,11/15/2019 12:37:50 PM,N51741,1,history
3,1,U80234,11/15/2019 12:37:50 PM,N53234,1,history
4,1,U80234,11/15/2019 12:37:50 PM,N11276,1,history


In [8]:
interactions_history.to_parquet("../data/processed/interactions_history_train.parquet")
interactions_impression.to_parquet("../data/processed/interactions_impression_train.parquet")

In [10]:
print("Train history rows:", len(interactions_history))
print("Train impression rows:", len(interactions_impression))
print("Clicked impression rows:", interactions_impression["click"].sum())

print("\nDev history rows:", len(interactions_history_dev))
print("Dev impression rows:", len(interactions_impression_dev))
print("Clicked impression rows:", interactions_impression_dev["click"].sum())

Train history rows: 5107639
Train impression rows: 5843444
Clicked impression rows: 236344

Dev history rows: 2362514
Dev impression rows: 2740998
Clicked impression rows: 111383


In [14]:
malformed_count = 0

impression_rows = behaviors.apply(parse_impressions, axis=1).explode().dropna()
print("Train malformed impression tokens:", malformed_count)

malformed_count = 0

impression_rows_dev = behaviors_dev.apply(parse_impressions, axis=1).explode().dropna()
print("Dev malformed impression tokens:", malformed_count)

Train malformed impression tokens: 0
Dev malformed impression tokens: 0


In [16]:
train_news = pd.read_csv(
    "../data/raw/MINDsmall_train/news.tsv",
    sep="\t", header=None,
    names=["news_id", "category", "subcategory", "title", "abstract",
           "url", "title_entities", "abstract_entities"]
)

dev_news = pd.read_csv(
    "../data/raw/MINDsmall_dev/news.tsv",
    sep="\t", header=None,
    names=["news_id", "category", "subcategory", "title", "abstract",
           "url", "title_entities", "abstract_entities"]
)

news_all = pd.concat([train_news, dev_news], ignore_index=True)

In [17]:
dupe_check = (
    news_all.groupby("news_id")[["category", "subcategory", "title"]]
    .nunique()
)

inconsistent_news = dupe_check[
    (dupe_check["category"] > 1)
    | (dupe_check["subcategory"] > 1)
    | (dupe_check["title"] > 1)
]

print("News items with inconsistent metadata across splits:", len(inconsistent_news))

News items with inconsistent metadata across splits: 0


In [19]:
news_all = news_all.drop_duplicates(subset=["news_id"])

print("Combined news shape:", news_all.shape)
assert news_all["news_id"].is_unique
display(news_all.head())

Combined news shape: (65238, 8)


,news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


In [ ]:
news_all.to_parquet("../data/processed/news.parquet")

In [21]:
train_with_news = interactions_history.merge(
    news_all[["news_id", "category", "subcategory", "title", "abstract"]],
    left_on="item_id", right_on="news_id", how="left"
)

dev_with_news = interactions_history_dev.merge(
    news_all[["news_id", "category", "subcategory", "title", "abstract"]],
    left_on="item_id", right_on="news_id", how="left"
)

missing_news = train_with_news["category"].isna().sum()
print("Train interactions with no matching news metadata:", missing_news)
missing_news = dev_with_news["category"].isna().sum()
print("Dev interactions with no matching news metadata:", missing_news)

Train interactions with no matching news metadata: 0
Dev interactions with no matching news metadata: 0


In [22]:
train_with_news.to_parquet("../data/processed/train_with_news.parquet")
dev_with_news.to_parquet("../data/processed/dev_with_news.parquet")